# v1 Sparse Retrieval

Dense (Qdrant/Ollama embeddings) + sparse (BM25) retrieval over the seeded doc set, combined via Reciprocal Rank Fusion (RRF).

In [1]:
import sys
sys.path.insert(0, "..")

import asyncio
from rank_bm25 import BM25Okapi

from app.services.ollama_service import embed
from app.services.qdrant.factory import make_qdrant_service

/Users/michaeleko/Documents/Works/ariapay/ariabot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load corpus from Qdrant

Pull every seeded point (source/heading/text) so BM25 has a corpus and dense search has a baseline to compare against.

In [2]:
service = make_qdrant_service()

points = []
offset = None
while True:
    batch, offset = await service.client.scroll(
        service.collection_name, limit=1000, offset=offset, with_payload=True, with_vectors=False
    )
    points.extend(batch)
    if offset is None:
        break
docs = [
    {
        "id": p.id,
        "source": p.payload["source"],
        "heading": p.payload["heading"],
        "text": p.payload["text"],
    }
    for p in points
]
print(f"{len(docs)} docs loaded")
docs[0]

28 docs loaded


{'id': '014ed301-1cb9-8c7b-dad5-42fff6f0e82e',
 'source': 'privacy.md',
 'heading': "8. Children's Privacy",
 'text': 'Ariapay is not intended for use by anyone under 18 years of age, and we do not knowingly collect personal data from children.'}

## Sparse retrieval (BM25)

Sparse search (BM25) — classic keyword search. Scores docs by term overlap + rarity (rare shared words score higher, common words discounted). Catches exact term matches dense sometimes misses.



In [3]:
def tokenize(text: str) -> list[str]:
    return text.lower().split()

corpus_tokens = [tokenize(f"{d['heading']} {d['text']}") for d in docs]
bm25 = BM25Okapi(corpus_tokens)

def sparse_search(query: str, top_k: int = 10) -> list[tuple[int, float]]:
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return [(i, scores[i]) for i in ranked[:top_k] if scores[i] > 0]

## Dense retrieval (Qdrant + Ollama embeddings)

Dense search — query text → embedding vector (via Ollama, 4096 dims) → Qdrant finds closest doc vectors by cosine similarity. Catches semantic/meaning matches even w/ different wording ("kids" ~ "children").



In [4]:
id_to_idx = {d["id"]: i for i, d in enumerate(docs)}

async def dense_search(query: str, top_k: int = 10) -> list[tuple[int, float]]:
    vector = await embed(query)
    hits = await service.client.query_points(
        collection_name=service.collection_name,
        query=vector,
        limit=top_k,
    )
    return [(id_to_idx[h.id], h.score) for h in hits.points if h.id in id_to_idx]

## RRF fusion

`score(d) = sum over rankers of 1 / (k + rank(d))`, rank is 1-indexed. Standard `k=60`.

RRF fusion — run both, get two ranked lists (top 20 each). For each doc, score = sum of 1/(60+rank) across whichever list(s) it appears in. Doc ranked #1 in both lists beats doc ranked #1 in only one. No need to normalize/compare raw scores (BM25 scores and cosine scores aren't on same scale) — RRF only cares about rank position, sidesteps that problem entirely.

Final top-5 = fused ranking, best of both worlds: semantic recall + exact keyword precision.



In [5]:
def rrf_fuse(rankings: list[list[tuple[int, float]]], k: int = 60) -> list[tuple[int, float]]:
    scores: dict[int, float] = {}
    for ranking in rankings:
        for rank, (idx, _) in enumerate(ranking, start=1):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)

In [6]:
async def hybrid_search(query: str, top_k: int = 5):
    sparse_hits = sparse_search(query, top_k=20)
    dense_hits = await dense_search(query, top_k=20)
    fused = rrf_fuse([dense_hits, sparse_hits])[:top_k]

    print(f"Query: {query!r}\n")
    print("-- dense top --")
    for idx, score in dense_hits[:5]:
        d = docs[idx]
        print(f"  {score:.4f}  {d['source']} / {d['heading']}")

    print("\n-- sparse (BM25) top --")
    for idx, score in sparse_hits[:5]:
        d = docs[idx]
        print(f"  {score:.4f}  {d['source']} / {d['heading']}")

    print("\n-- RRF fused top --")
    for idx, score in fused:
        d = docs[idx]
        print(f"  {score:.4f}  {d['source']} / {d['heading']}")

    return fused

## Try it

In [7]:
_ = await hybrid_search("how do you handle children's privacy")

Query: "how do you handle children's privacy"

-- dense top --
  0.5836  privacy.md / 8. Children's Privacy
  0.5531  privacy.md / 10. Contact Us
  0.5297  privacy.md / 9. Changes to This Policy
  0.5117  privacy.md / 7. Cookies and Tracking
  0.5099  privacy.md / 3. Data Sharing and Disclosure

-- sparse (BM25) top --
  8.3727  privacy.md / 10. Contact Us
  7.3350  privacy.md / 8. Children's Privacy
  3.1022  terms.md / 8. Data Privacy
  2.6231  terms.md / 1. Acceptance of Terms
  2.4923  privacy.md / 7. Cookies and Tracking

-- RRF fused top --
  0.0325  privacy.md / 8. Children's Privacy
  0.0325  privacy.md / 10. Contact Us
  0.0310  privacy.md / 7. Cookies and Tracking
  0.0306  privacy.md / 9. Changes to This Policy
  0.0303  privacy.md / 


In [8]:
_ = await hybrid_search("what cookies and tracking technologies are used")

Query: 'what cookies and tracking technologies are used'

-- dense top --
  0.7529  privacy.md / 7. Cookies and Tracking
  0.5365  privacy.md / 2. How We Use Your Information
  0.5342  privacy.md / 1. Information We Collect
  0.5174  privacy.md / 10. Contact Us
  0.5144  privacy.md / 4. Data Security

-- sparse (BM25) top --
  13.0550  privacy.md / 7. Cookies and Tracking
  4.0995  about.md / What we stand for
  2.7742  privacy.md / 4. Data Security
  2.0980  terms.md / 2. Description of Service
  1.8924  terms.md / 8. Data Privacy

-- RRF fused top --
  0.0328  privacy.md / 7. Cookies and Tracking
  0.0313  privacy.md / 4. Data Security
  0.0302  privacy.md / 2. How We Use Your Information
  0.0298  privacy.md / 1. Information We Collect
  0.0297  terms.md / 8. Data Privacy


In [9]:
_ = await hybrid_search("data privacy and personal information")

Query: 'data privacy and personal information'

-- dense top --
  0.5783  privacy.md / 3. Data Sharing and Disclosure
  0.5611  privacy.md / 10. Contact Us
  0.5482  privacy.md / 6. Your Rights
  0.5408  terms.md / 8. Data Privacy
  0.5357  privacy.md / 5. Data Retention

-- sparse (BM25) top --
  7.2723  terms.md / 8. Data Privacy
  6.0243  privacy.md / 8. Children's Privacy
  5.6245  privacy.md / 3. Data Sharing and Disclosure
  4.7011  privacy.md / 5. Data Retention
  4.1009  privacy.md / 1. Information We Collect

-- RRF fused top --
  0.0323  privacy.md / 3. Data Sharing and Disclosure
  0.0320  terms.md / 8. Data Privacy
  0.0310  privacy.md / 5. Data Retention
  0.0308  privacy.md / 6. Your Rights
  0.0304  privacy.md / 8. Children's Privacy


## Late-interaction reranking (ColBERT, via Qdrant native multivector)

Rerank the RRF-fused candidates with a ColBERT-style late-interaction model. Each doc/query gets one vector *per token* instead of one pooled vector; relevance is MaxSim — for every query token, take its best match among the doc's tokens, then sum. Captures finer-grained term interactions than a single dense vector or BM25 alone.

Scored using Qdrant's own local multivector engine (an in-memory collection with `MultiVectorConfig` / `MAX_SIM` comparator) — the same scoring path Qdrant uses server-side for native late-interaction search — rather than hand-rolled MaxSim math. Only the fused top-N candidates get ColBERT vectors computed, not the whole corpus, so it stays cheap.

In [10]:
from fastembed import LateInteractionTextEmbedding
from qdrant_client import QdrantClient, models as qm

colbert = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")
COLBERT_DIM = 128


def rerank_late_interaction(query: str, candidate_idxs: list[int], top_k: int = 5) -> list[tuple[int, float]]:
    if not candidate_idxs:
        return []

    query_vec = next(colbert.query_embed(query)).tolist()
    doc_texts = [f"{docs[i]['heading']}\n\n{docs[i]['text']}" for i in candidate_idxs]
    doc_vecs = [v.tolist() for v in colbert.embed(doc_texts)]

    rr = QdrantClient(":memory:")
    rr.create_collection(
        "rerank",
        vectors_config=qm.VectorParams(
            size=COLBERT_DIM,
            distance=qm.Distance.COSINE,
            multivector_config=qm.MultiVectorConfig(comparator=qm.MultiVectorComparator.MAX_SIM),
        ),
    )
    rr.upsert(
        "rerank",
        points=[qm.PointStruct(id=i, vector=vec) for i, vec in enumerate(doc_vecs)],
    )
    hits = rr.query_points("rerank", query=query_vec, limit=top_k).points
    return [(candidate_idxs[h.id], h.score) for h in hits]


async def hybrid_search_reranked(query: str, top_k: int = 5, candidate_pool: int = 20):
    sparse_hits = sparse_search(query, top_k=candidate_pool)
    dense_hits = await dense_search(query, top_k=candidate_pool)
    fused = rrf_fuse([dense_hits, sparse_hits])
    fused_idxs = [idx for idx, _ in fused[:candidate_pool]]

    reranked = rerank_late_interaction(query, fused_idxs, top_k=top_k)

    print(f"Query: {query!r}\n")
    print("-- RRF fused top (pre-rerank) --")
    for idx, score in fused[:top_k]:
        d = docs[idx]
        print(f"  {score:.4f}  {d['source']} / {d['heading']}")

    print("\n-- reranked (ColBERT late-interaction) top --")
    for idx, score in reranked:
        d = docs[idx]
        print(f"  {score:.4f}  {d['source']} / {d['heading']}")

    return reranked

Fetching 5 files: 100%|██████████| 5/5 [00:10<00:00,  2.17s/it]


## Try it (reranked) — same queries, compare against the fused-only results above

In [11]:
_ = await hybrid_search_reranked("how do you handle children's privacy")

Query: "how do you handle children's privacy"

-- RRF fused top (pre-rerank) --
  0.0325  privacy.md / 8. Children's Privacy
  0.0325  privacy.md / 10. Contact Us
  0.0310  privacy.md / 7. Cookies and Tracking
  0.0306  privacy.md / 9. Changes to This Policy
  0.0303  privacy.md / 

-- reranked (ColBERT late-interaction) top --
  20.2427  privacy.md / 8. Children's Privacy
  15.2348  privacy.md / 10. Contact Us
  13.7068  terms.md / 8. Data Privacy
  12.5196  terms.md / 1. Acceptance of Terms
  11.0571  privacy.md / 2. How We Use Your Information


In [12]:
_ = await hybrid_search_reranked("what cookies and tracking technologies are used")

Query: 'what cookies and tracking technologies are used'

-- RRF fused top (pre-rerank) --
  0.0328  privacy.md / 7. Cookies and Tracking
  0.0313  privacy.md / 4. Data Security
  0.0302  privacy.md / 2. How We Use Your Information
  0.0298  privacy.md / 1. Information We Collect
  0.0297  terms.md / 8. Data Privacy

-- reranked (ColBERT late-interaction) top --
  25.8842  privacy.md / 7. Cookies and Tracking
  11.7946  terms.md / 2. Description of Service
  9.1117  privacy.md / 4. Data Security
  8.6638  privacy.md / 1. Information We Collect
  8.5549  terms.md / 8. Data Privacy


In [13]:
_ = await hybrid_search_reranked("data privacy and personal information")

Query: 'data privacy and personal information'

-- RRF fused top (pre-rerank) --
  0.0323  privacy.md / 3. Data Sharing and Disclosure
  0.0320  terms.md / 8. Data Privacy
  0.0310  privacy.md / 5. Data Retention
  0.0308  privacy.md / 6. Your Rights
  0.0304  privacy.md / 8. Children's Privacy

-- reranked (ColBERT late-interaction) top --
  23.9836  terms.md / 8. Data Privacy
  19.9921  privacy.md / 3. Data Sharing and Disclosure
  18.7684  privacy.md / 5. Data Retention
  17.4707  privacy.md / 8. Children's Privacy
  16.0504  privacy.md / 6. Your Rights
